# Sparse Retrieval — when keyword search wins, and when it fails

**Notebook 06 · Phase 2 (Retrieval foundations)** · Stack: `bm25s`, `sentence-transformers` (SPLADE), `langchain-openai`

Dense embeddings match on **meaning**; sparse retrieval matches on the **terms a document
literally contains**. Neither is universally best. This notebook *proves, with a labeled
multi-domain corpus*, exactly **which strategy wins on which kind of query** — so you know when
to reach for each (and why production fuses them, Notebook 07).

We compare three retrievers head-to-head on every query, against **known-correct answers**:

- **BM25** (classical sparse) — `bm25s`
- **Dense** (semantic) — OpenAI `text-embedding-3-small`
- **SPLADE** (learned sparse) — `naver/splade-cocondenser-ensembledistil`

Then we go deep on BM25 mechanics, its production variants, and SPLADE's learned expansion.

> **Myth we'll correct with evidence:** "embeddings can't do exact codes." Modern embeddings are
> actually decent at them. The *reproducible* truth is about **decisiveness, cost, and blind
> spots** — which the comparison below makes concrete.

## 0. Install dependencies

Run first (idempotent). Restart the kernel once after a fresh install. *(SPLADE downloads a
~500 MB model on first use; it runs on CPU here.)*

In [1]:
%pip install -q \
    "numpy<2" \
    bm25s PyStemmer scikit-learn \
    sentence-transformers \
    langchain-openai python-dotenv
print("✅ Dependencies ready. If this was a fresh install, restart the kernel, then re-run.")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Dependencies ready. If this was a fresh install, restart the kernel, then re-run.


## 1. Setup — load all three retrievers

In [3]:
import warnings, os
warnings.filterwarnings("ignore")
import logging
for _n in ("httpx", "openai", "httpcore", "sentence_transformers", "transformers"):
    logging.getLogger(_n).setLevel(logging.ERROR)
from pathlib import Path
from dotenv import load_dotenv
import numpy as np

load_dotenv(Path.cwd().parent / ".env")

from langchain_openai import OpenAIEmbeddings
import bm25s
from Stemmer import Stemmer
from sentence_transformers import SparseEncoder

dense_model = OpenAIEmbeddings(model="text-embedding-3-small")
stemmer = Stemmer("english")
# device="cpu": Apple's MPS backend lacks the sparse-CSR op SPLADE's decode()/sparsity() use.
splade = SparseEncoder("naver/splade-cocondenser-ensembledistil", device="cpu")

def cos(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def tok(texts):
    return bm25s.tokenize(texts, stopwords="en", stemmer=stemmer.stemWords, return_ids=False, show_progress=False)

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("Loaded: BM25 (bm25s) · Dense (OpenAI 3-small) · SPLADE (splade-cocondenser)")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

OPENAI_API_KEY set: True
Loaded: BM25 (bm25s) · Dense (OpenAI 3-small) · SPLADE (splade-cocondenser)


## 2. A labeled, multi-domain corpus

Sixteen short documents across **software, legal, medical, science, e-commerce, finance**. Each
test query below has a **known gold answer** (the one document that truly answers it), so we can
*verify* whether each retriever actually finds the right thing — not just eyeball it.

In [ ]:
corpus = [
    # software / errors
    "Error ERR_CONN_TIMEOUT occurs when the database connection exceeds the 30 second limit. Increase the pool timeout to resolve it.",  # 0
    "Error ERR_CONN_TIMEOUT_DB occurs when the database connection exceeds the 90 second limit. Check the network configuration and try again.",   # 1",
    "Size the database connection pool to your workload and monitor active connections regularly.",                                      # 1
    # legal / contracts
    "Either party may terminate this agreement by giving sixty days written notice to the other party.",                                 # 2
    "To cancel a product order, contact customer support within 24 hours of purchase.",                                                  # 3
    # medical
    "Myocardial infarction results from coronary artery occlusion, causing ischemic damage to cardiac muscle.",                          # 4
    "A stroke occurs when the blood supply to part of the brain is interrupted or reduced.",                                             # 5
    # science
    "Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen within plant leaves.",                          # 6
    "Our food processing plant makes packaged snacks and ready meals for retail customers.",                                             # 7
    # e-commerce
    "Product SKU7788XL is the extra-large blue hoodie, priced at 49 dollars.",  
    "Product SKU7799XL is the extra-small baby hoodie, priced at 9 dollars.",                                                           # 8
    "The extra-large hoodie is available in blue, black, and grey while stocks last.",                                                   # 9
    # finance
    "A 401k is a tax advantaged retirement savings plan sponsored by an employer.",                                                      # 10
    "An IRA is an individual retirement account with annual contribution limits.",                                                       # 11
    # software / versions
    "Python 3.11 introduced fine-grained error locations in tracebacks and significantly faster startup.",                              # 12
    "Python 3.10 introduced structural pattern matching with the new match statement.",                                                 # 13
    "Python 3.12 introduced improved f-strings and a per-interpreter GIL for isolation.",                                               # 14
    "The pandas function df.merge joins two DataFrames on a key column.",                                                               # 15
]

# (name, query, gold_doc_id, archetype)
QUERIES = [
    ("exact code",        "ERR_CONN_TIMEOUT",                       0,  "exact token"),
    ("synonym gap",       "How do I end my contract early?",        3,  "vocabulary gap"),
    ("paraphrase",        "How do plants make food using light?",   7,  "semantic"),
    ("exact SKU",         "SKU7788XL",                              9,  "exact token"),
    ("lay -> technical",  "what is a heart attack?",                5,  "vocabulary gap"),
    ("version number",    "Python 3.11 new features",              13,  "exact token (fragments)"),
]
print(len(corpus), "docs |", len(QUERIES), "gold-labeled queries")

17 docs | 6 gold-labeled queries


### Index the corpus once with each retriever

Same corpus, three representations: BM25 term counts, dense vectors, SPLADE sparse vectors.

In [28]:
bm25 = bm25s.BM25(method="lucene")
bm25.index(tok(corpus), show_progress=True)

BM25S Create Vocab:   0%|          | 0/17 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/17 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/17 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/17 [00:00<?, ?it/s]

In [29]:
bm25.vocab_dict
bm25.build_index_from_ids

<bound method BM25.build_index_from_ids of <bm25s.BM25 object at 0x33cb1acd0>>

In [30]:
import numpy as np, pandas as pd
from scipy.sparse import csc_matrix

data   = bm25.scores["data"]
indices = bm25.scores["indices"]
indptr = bm25.scores["indptr"]

n_terms = len(indptr) - 1
n_docs  = len(corpus)

# columns = terms, rows = docs
M = csc_matrix((data, indices, indptr), shape=(n_docs, n_terms))

vocab = {v: k for k, v in bm25.vocab_dict.items()}
cols = [vocab.get(i, f"<{i}>") for i in range(n_terms)]

df = pd.DataFrame(M.toarray(), columns=cols)
df.to_csv("out.csv", index=False)



In [31]:
import pandas as pd, torch
from pathlib import Path
splade_docs = splade.encode_document(corpus)
M = splade_docs.to_dense() if splade_docs.is_sparse else splade_docs
M = M.cpu().numpy()

tokenizer = splade.tokenizer          # or splade[0].tokenizer
vocab = tokenizer.convert_ids_to_tokens(range(M.shape[1]))

df = pd.DataFrame(M, columns=vocab)
df = df.loc[:, (df != 0).any(axis=0)]   # drop all-zero columns


df.to_csv('splade_output.csv', index=False)
print(df.shape)

(17, 1072)


In [36]:
# BM25
bm25 = bm25s.BM25(method="lucene")
bm25.index(tok(corpus), show_progress=True)
# Dense (OpenAI)
dense_vecs = dense_model.embed_documents(corpus)
# SPLADE
splade_docs = splade.encode_document(corpus)

def dense_scores(q):  return np.array([cos(dense_model.embed_query(q), v) for v in dense_vecs])
def bm25_scores(q):   return bm25.get_scores(tok(q)[0])
def splade_scores(q): return splade.similarity(splade.encode_query([q]), splade_docs).numpy().ravel()

def topk(scores, k=3):
    order = list(np.argsort(scores)[::-1])[:k]
    return [(i, float(scores[i])) for i in order]
print("indexed with all three retrievers ✓")

BM25S Create Vocab:   0%|          | 0/17 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/17 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/17 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/17 [00:00<?, ?it/s]

indexed with all three retrievers ✓


## 3. Head-to-head by query type

`compare()` runs all three retrievers on a query and marks whether each put the **gold** document
at rank 1. Watch the pattern emerge across archetypes.

In [37]:
def compare(name, query, gold, archetype):
    print(f"[{archetype}]  {name}:  {query!r}   (gold = doc {gold})")
    for label, scores in [("BM25 ", bm25_scores(query)),
                          ("Dense", dense_scores(query)),
                          ("SPLADE", splade_scores(query))]:
        order = list(np.argsort(scores)[::-1])
        rank = order.index(gold) + 1
        top = order[0]
        flag = "✅ HIT " if rank == 1 else f"❌ miss#{rank}"
        print(f"   {label:6} {flag}  top1=doc{top} (score {scores[top]:.3f})  ->  {corpus[top][:52]}")
    print()

### 3a. Exact-token queries — BM25's home turf

An error code and a product SKU. BM25 gives the rare token a huge IDF weight → a **decisive**
score (and nothing else scores at all). Dense also finds them, but as a fuzzier signal.

In [38]:
compare(*QUERIES[0])   # exact code
compare(*QUERIES[3])   # exact SKU

[exact token]  exact code:  'ERR_CONN_TIMEOUT'   (gold = doc 0)
   BM25   ✅ HIT   top1=doc0 (score 0.858)  ->  Error ERR_CONN_TIMEOUT occurs when the database conn
   Dense  ❌ miss#2  top1=doc1 (score 0.592)  ->  Error ERR_CONN_TIMEOUT_DB occurs when the database c
   SPLADE ✅ HIT   top1=doc0 (score 26.338)  ->  Error ERR_CONN_TIMEOUT occurs when the database conn

[exact token]  exact SKU:  'SKU7788XL'   (gold = doc 9)
   BM25   ✅ HIT   top1=doc9 (score 1.056)  ->  Product SKU7788XL is the extra-large blue hoodie, pr
   Dense  ✅ HIT   top1=doc9 (score 0.532)  ->  Product SKU7788XL is the extra-large blue hoodie, pr
   SPLADE ✅ HIT   top1=doc9 (score 17.077)  ->  Product SKU7788XL is the extra-large blue hoodie, pr



### 3b. Vocabulary-gap queries — BM25's blind spot

The query and the gold document share **no words** ("end my contract" vs "terminate this
agreement"; "heart attack" vs "myocardial infarction"). BM25 literally cannot see the answer —
the gold document scores **0.0** — while dense and SPLADE bridge the gap semantically.

In [7]:
compare(*QUERIES[1])   # synonym gap
compare(*QUERIES[4])   # lay -> technical

[vocabulary gap]  synonym gap:  'How do I end my contract early?'   (gold = doc 2)


   BM25   ❌ miss#14  top1=doc15 (score 0.000)  ->  The pandas function df.merge joins two DataFrames on
   Dense  ✅ HIT   top1=doc2 (score 0.441)  ->  Either party may terminate this agreement by giving 
   SPLADE ✅ HIT   top1=doc2 (score 10.395)  ->  Either party may terminate this agreement by giving 

[vocabulary gap]  lay -> technical:  'what is a heart attack?'   (gold = doc 4)


   BM25   ❌ miss#12  top1=doc15 (score 0.000)  ->  The pandas function df.merge joins two DataFrames on
   Dense  ✅ HIT   top1=doc4 (score 0.418)  ->  Myocardial infarction results from coronary artery o
   SPLADE ✅ HIT   top1=doc4 (score 7.949)  ->  Myocardial infarction results from coronary artery o



### 3c. Paraphrase with a lexical lure

"plants make food using light" shares surface words with a *food-processing plant* distractor.
BM25 takes the bait; dense/SPLADE understand it's about **photosynthesis**.

In [8]:
compare(*QUERIES[2])

[semantic]  paraphrase:  'How do plants make food using light?'   (gold = doc 6)


   BM25   ❌ miss#2  top1=doc7 (score 2.600)  ->  Our food processing plant makes packaged snacks and 
   Dense  ✅ HIT   top1=doc6 (score 0.567)  ->  Photosynthesis converts sunlight, water, and carbon 
   SPLADE ✅ HIT   top1=doc6 (score 12.517)  ->  Photosynthesis converts sunlight, water, and carbon 



### 3d. Version/number exactness — BM25 fragments the token

`bm25s` splits `3.11` into `3` / `11`; "3" is common to every version doc, so BM25 drifts to the
wrong release. Dense/SPLADE keep the version meaning intact.

In [9]:
compare(*QUERIES[5])
print("why BM25 drifts — tokenization:", tok("Python 3.11")[0])

[exact token (fragments)]  version number:  'Python 3.11 new features'   (gold = doc 12)


   BM25   ❌ miss#2  top1=doc13 (score 1.683)  ->  Python 3.10 introduced structural pattern matching w
   Dense  ✅ HIT   top1=doc12 (score 0.676)  ->  Python 3.11 introduced fine-grained error locations 
   SPLADE ✅ HIT   top1=doc12 (score 17.203)  ->  Python 3.11 introduced fine-grained error locations 

why BM25 drifts — tokenization: ['python', '11']


## 4. The full comparison matrix

One grid, all queries. `HIT` = gold ranked #1. The **margin** column for Dense is the gap between
its #1 and #2 scores — a small margin means "technically right, but fragile."

In [10]:
def rank_and_margin(scores, gold):
    order = list(np.argsort(scores)[::-1]); rank = order.index(gold) + 1
    s = np.sort(scores)[::-1]; margin = float(s[0] - s[1])
    return rank, margin

hdr = f"{'query':18} {'archetype':22} {'BM25':7} {'Dense':16} {'SPLADE':7}"
print(hdr); print("-" * len(hdr))
for name, q, gold, arch in QUERIES:
    br, _  = rank_and_margin(bm25_scores(q), gold)
    dr, dm = rank_and_margin(dense_scores(q), gold)
    sr, _  = rank_and_margin(splade_scores(q), gold)
    b = "HIT" if br == 1 else f"miss#{br}"
    d = ("HIT" if dr == 1 else f"miss#{dr}") + f" (m={dm:.2f})"
    s = "HIT" if sr == 1 else f"miss#{sr}"
    print(f"{name:18} {arch:22} {b:7} {d:16} {s:7}")

query              archetype              BM25    Dense            SPLADE 
--------------------------------------------------------------------------


exact code         exact token            HIT     HIT (m=0.29)     HIT    


synonym gap        vocabulary gap         miss#14 HIT (m=0.13)     HIT    


paraphrase         semantic               miss#2  HIT (m=0.31)     HIT    


exact SKU          exact token            HIT     HIT (m=0.27)     HIT    


lay -> technical   vocabulary gap         miss#12 HIT (m=0.06)     HIT    


version number     exact token (fragments) miss#2  HIT (m=0.11)     HIT    


**Read the grid:**

- **BM25** wins *exact tokens* decisively and cheaply, but **misses every vocabulary-gap and
  paraphrase query** — and fragments version numbers. Its blind spot is real and total (score 0).
- **Dense** finds everything here (modern embeddings are strong) — but note the **thin margins**
  on the gap queries (e.g. ~0.05). "Technically #1" by a hair is fragile once your corpus grows to
  millions of near-duplicates.
- **SPLADE** hits everything: exact-term precision *and* learned semantic expansion — the best
  single retriever, at a higher compute cost than BM25.

Nobody is free: BM25 is blind to meaning, dense is fuzzy and costs an embed call per doc/query,
SPLADE needs a GPU-ish model. This is precisely why production runs **hybrid** (Notebook 07).

## 5. Under the hood — BM25 mechanics (`k1`, `b`)

BM25 scores a document by its query terms, weighted by:
- **IDF** — rare terms (an error code) outweigh common ones ("the").
- **TF saturation (`k1`)** — repeated terms have *diminishing returns* (unlike linear TF-IDF).
- **Length normalization (`b`)** — long docs don't win just for being long. `b=0` off, `b=1` full.

Defaults: **`k1` ≈ 1.2–2.0, `b` = 0.75**. Watch the gold score for the error-code query shift
with `b`:

In [11]:
q = "ERR_CONN_TIMEOUT"
for b in [0.0, 0.75, 1.0]:
    r = bm25s.BM25(method="lucene", k1=1.5, b=b)
    r.index(tok(corpus), show_progress=False)
    print(f"b={b:>4}: gold(doc0) score = {r.get_scores(tok(q)[0])[0]:.3f}   (k1=1.5)")

b= 0.0: gold(doc0) score = 0.971   (k1=1.5)
b=0.75: gold(doc0) score = 0.826   (k1=1.5)
b= 1.0: gold(doc0) score = 0.786   (k1=1.5)


## 6. BM25 variants — *which* BM25 do you mean?

"BM25" is a family. Implementations differ in IDF form and TF treatment, so the **same query
scores differently** across libraries/engines — a real reproducibility gotcha
([Kamphuis et al., 2020](https://link.springer.com/chapter/10.1007/978-3-030-45442-5_4)).

| Variant | What it changes |
|---------|-----------------|
| **Lucene** | What Elasticsearch/OpenSearch/Solr run — the de-facto "BM25". |
| **Robertson** | Original Okapi IDF (can go negative for very common terms). |
| **ATIRE** | Cleaner IDF that avoids negatives. |
| **BM25+** | Adds a `δ` lower bound so long docs keep a floor score. |
| **BM25L** | Reshapes length normalization to treat long docs fairly. |
| **BM25F** *(engine-only)* | Fielded: weights title/body/anchor separately (needs Elasticsearch/Solr). |

In [12]:
q = "SKU7788XL"
print(f"Query {q!r} — gold is doc 8. Rankings agree; SCORES do not:\n")
for method in ["lucene", "robertson", "atire", "bm25+", "bm25l"]:
    r = bm25s.BM25(method=method); r.index(tok(corpus), show_progress=False)
    order = list(np.argsort(r.get_scores(tok(q)[0]))[::-1])
    print(f"  {method:9s} top1=doc{order[0]}  gold_score={r.get_scores(tok(q)[0])[8]:.3f}")
print("\n→ Never compare raw scores across variants/engines — only rankings within one scorer.")

Query 'SKU7788XL' — gold is doc 8. Rankings agree; SCORES do not:

  lucene    top1=doc8  gold_score=1.020
  robertson top1=doc8  gold_score=0.981
  atire     top1=doc8  gold_score=2.911
  bm25+     top1=doc8  gold_score=4.391
  bm25l     top1=doc8  gold_score=3.119

→ Never compare raw scores across variants/engines — only rankings within one scorer.


## 7. Learned sparse — inside SPLADE

Why did SPLADE hit *every* query in Section 4? Because it emits a **sparse vector over the whole
vocabulary**, weighting relevant terms *including ones not in the text* — learned **expansion**.
It keeps BM25's inverted-index efficiency while adding semantic bridging.

*(Cousins: **doc2query/docT5query** append generated queries before BM25 indexing; **uniCOIL**
and **DeepImpact** learn a scalar weight per token.)*

In [13]:
doc = corpus[0]  # the ERR_CONN_TIMEOUT document
emb = splade.encode_document([doc])
print("doc:", doc[:70], "...")
print("SPLADE vector shape:", tuple(emb.shape), "(= vocabulary size)")
print(f"sparsity: {SparseEncoder.sparsity(emb)['sparsity_ratio']:.2%}  (only a few dims are non-zero)")
print("\nTop weighted tokens — several are EXPANSION terms never written in the doc:")
top = splade.decode(emb, top_k=14)[0]
print("  " + ", ".join(f"{t.strip()}={v:.2f}" for t, v in top))

doc: Error ERR_CONN_TIMEOUT occurs when the database connection exceeds the ...
SPLADE vector shape: (1, 30522) (= vocabulary size)
sparsity: 99.66%  (only a few dims are non-zero)

Top weighted tokens — several are EXPANSION terms never written in the doc:
  er=2.45, con=1.98, ##out=1.75, error=1.69, time=1.49, database=1.48, pool=1.42, errors=1.34, ##r=1.33, limit=1.27, timing=1.27, ##n=1.26, occur=1.19, connection=1.16


## 8. Where sparse fits — hybrid + Reciprocal Rank Fusion

Since BM25 and dense have *complementary* blind spots, production runs **both** and fuses the
ranked lists with **RRF** — robust because it uses ranks, not raw scores:

`RRF(d) = Σ 1 / (k + rank_i(d))`   (typically `k=60`).

When the retrievers **agree** (e.g. an exact-token query both rank #1), RRF cleanly keeps the gold
document on top:

In [14]:
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

q, gold = "ERR_CONN_TIMEOUT", 0
bm25_order  = list(np.argsort(bm25_scores(q))[::-1])
dense_order = list(np.argsort(dense_scores(q))[::-1])
fused = rrf([bm25_order, dense_order])
print(f"query: {q!r}  (gold = doc {gold})")
print(f"   BM25 gold #{bm25_order.index(gold)+1} | Dense gold #{dense_order.index(gold)+1} | "
      f"RRF gold #{fused.index(gold)+1}  ->  {corpus[fused[0]][:48]}")

# Honest caveat: naive full-list RRF is NOT a silver bullet.
q2, gold2 = "How do I end my contract early?", 2   # BM25 is blind here (scored gold 0.0)
b2 = list(np.argsort(bm25_scores(q2))[::-1]); d2 = list(np.argsort(dense_scores(q2))[::-1])
f2 = rrf([b2, d2])
print(f"\nBUT when one retriever is blind ({q2!r}):")
print(f"   BM25 gold #{b2.index(gold2)+1} (blind) | Dense gold #{d2.index(gold2)+1} | "
      f"naive RRF gold #{f2.index(gold2)+1}  <- dragged down by BM25 noise")

query: 'ERR_CONN_TIMEOUT'  (gold = doc 0)
   BM25 gold #1 | Dense gold #1 | RRF gold #1  ->  Error ERR_CONN_TIMEOUT occurs when the database 



BUT when one retriever is blind ('How do I end my contract early?'):
   BM25 gold #14 (blind) | Dense gold #1 | naive RRF gold #4  <- dragged down by BM25 noise


**The honest takeaway.** RRF shines when retrievers agree or both carry signal. But naive
full-list fusion can be *dragged down* when one retriever is blind to a query (BM25 on a pure
vocabulary gap) or a strong lexical lure exists. Real hybrid systems therefore fuse only the
**top-K** from each retriever, apply **weights**, and finish with a **cross-encoder reranker** for
top-of-list precision. That complete pipeline is **Notebook 07**.

## 9. Summary — when to use which

| Query type | Best strategy | Why |
|------------|---------------|-----|
| Exact ID / code / SKU / citation | **BM25** (or SPLADE) | decisive, cheap, guaranteed exact match |
| Version / hyphenated IDs | **BM25 with care** + dense | tokenization fragments the token — test your analyzer |
| Synonyms / vocabulary gap | **Dense / SPLADE** | BM25 is blind when no words overlap (scores 0) |
| Paraphrase / semantic intent | **Dense / SPLADE** | meaning, not surface words |
| Mixed real-world traffic | **Hybrid (BM25 + Dense) + RRF** | complementary blind spots; the production standard |

### What we proved (on a labeled corpus)
- **BM25**: unbeatable and cheap on exact tokens; **totally blind** to vocabulary gaps (gold = 0.0).
- **Dense**: strong across the board, but **thin, fragile margins** on gap queries; costs an embed call.
- **SPLADE**: exact + learned expansion → best single retriever, at higher compute cost.
- **"BM25" is a family** — variants rank alike but score differently; never compare raw scores across engines.

### Next
**Notebook 07 — Hybrid search**: run dense + sparse together, fuse with **RRF**, then a
**cross-encoder reranker** for top-of-list precision, feeding a grounded, cited answer.